In [40]:
import pandas as pd
from pathlib import Path

In [41]:
BASE_DIR = Path.cwd().parent
RAW_DIR = BASE_DIR / "data_raw"

print([f.name for f in RAW_DIR.iterdir()])

['DS_MV_T_USD_Y_2025.xlsx', 'DS_RI_T_USD_M_2025.xlsx', 'Static_2025.xlsx', 'DS_MV_T_USD_M_2025.xlsx', 'DS_RI_T_USD_Y_2025.xlsx', 'DS_CO2_SCOPE_1_Y_2025.xlsx', 'DS_REV_Y_2025.xlsx', 'Risk_Free_Rate_2025.xlsx', 'DS_CO2_SCOPE_2_Y_2025.xlsx']


In [42]:
# Load scope1
scope1 = pd.read_excel(RAW_DIR / "DS_CO2_SCOPE_1_Y_2025.xlsx")
scope1 = scope1.dropna(subset=["ISIN"])
scope1 = scope1.replace("$$ER: E100,INVALID CODE OR EXPRESSION ENTERED", pd.NA)
scope1 = scope1.replace("$$ER: 0904,NO DATA AVAILABLE", pd.NA)
scope1 = scope1.drop(columns=[2025], errors="ignore")
year_cols = scope1.columns[2:]
scope1[year_cols] = scope1[year_cols].apply(pd.to_numeric, errors="coerce")
scope1[year_cols] = scope1[year_cols].T.ffill().T
scope1 = scope1.dropna(subset=year_cols, how="all")
scope1.to_excel("../data_clean/scope1_clean.xlsx", index=False)
print("scope1:", scope1.shape)

scope1: (2394, 28)


In [43]:
# Load scope2
scope2 = pd.read_excel(RAW_DIR / "DS_CO2_SCOPE_2_Y_2025.xlsx")
scope2 = scope2.dropna(subset=["ISIN"])
scope2 = scope2.replace("$$ER: E100,INVALID CODE OR EXPRESSION ENTERED", pd.NA)
scope2 = scope2.replace("$$ER: 0904,NO DATA AVAILABLE", pd.NA)
scope2 = scope2.drop(columns=[2025], errors="ignore")
year_cols = scope2.columns[2:]
scope2[year_cols] = scope2[year_cols].apply(pd.to_numeric, errors="coerce")
scope2[year_cols] = scope2[year_cols].T.ffill().T
scope2 = scope2.dropna(subset=year_cols, how="all")
scope2.to_excel("../data_clean/scope2_clean.xlsx", index=False)
print("scope2:", scope2.shape)

scope2: (2396, 28)


In [44]:
# Load revenues (divide by 1000 to convert thousands USD -> millions USD)
revenues = pd.read_excel(RAW_DIR / "DS_REV_Y_2025.xlsx")
revenues = revenues.dropna(subset=["ISIN"])
year_cols = revenues.columns[2:]
revenues[year_cols] = revenues[year_cols].apply(pd.to_numeric, errors="coerce")
revenues[year_cols] = revenues[year_cols].T.ffill().T
revenues = revenues.dropna(subset=year_cols, how="all")
revenues[year_cols] = revenues[year_cols] / 1000  # now in millions USD
revenues.to_excel("../data_clean/revenues_clean.xlsx", index=False)
print("revenues:", revenues.shape)

revenues: (2528, 29)


In [45]:
# Load annual market cap
mv_year = pd.read_excel(RAW_DIR / "DS_MV_T_USD_Y_2025.xlsx")
mv_year = mv_year.dropna(subset=["ISIN"])
year_cols = mv_year.columns[2:]
mv_year[year_cols] = mv_year[year_cols].apply(pd.to_numeric, errors="coerce")
mv_year[year_cols] = mv_year[year_cols].T.ffill().T
mv_year = mv_year.dropna(subset=year_cols, how="all")
mv_year.to_excel("../data_clean/market_value_year_clean.xlsx", index=False)
print("mv_year:", mv_year.shape)

mv_year: (2542, 29)


In [46]:
# Load monthly market cap
mv_month = pd.read_excel(RAW_DIR / "DS_MV_T_USD_M_2025.xlsx")
mv_month = mv_month.dropna(subset=["ISIN"])
mv_cols = mv_month.columns[2:]
mv_month[mv_cols] = mv_month[mv_cols].apply(pd.to_numeric, errors="coerce")
mv_month.to_excel("../data_clean/market_value_month_clean.xlsx", index=False)
print("mv_month:", mv_month.shape)

mv_month: (2545, 316)


In [47]:
import numpy as np

# Load monthly prices
prices = pd.read_excel(RAW_DIR / "DS_RI_T_USD_M_2025.xlsx")
prices = prices.dropna(subset=["ISIN"]).copy()
price_cols = prices.columns[2:]

# Force numeric
prices[price_cols] = prices[price_cols].apply(pd.to_numeric, errors="coerce")

# Treat prices below 0.5 as missing
prices[price_cols] = prices[price_cols].mask(prices[price_cols] < 0.5)

prices.head()

,NAME,ISIN,1999-12-31 00:00:00,2000-01-31 00:00:00,2000-02-29 00:00:00,2000-03-31 00:00:00,2000-04-28 00:00:00,2000-05-31 00:00:00,2000-06-30 00:00:00,2000-07-31 00:00:00,...,2025-04-30 00:00:00,2025-05-30 00:00:00,2025-06-30 00:00:00,2025-07-31 00:00:00,2025-08-29 00:00:00,2025-09-30 00:00:00,2025-10-31 00:00:00,2025-11-28 00:00:00,2025-12-31 00:00:00,2026-01-30 00:00:00
1,SLB,AN8068571086,1708.01,1858.26,2254.15,2334.75,2336.65,2245.09,2283.70,2262.67,...,3270.81,3251.13,3353.42,3353.42,3655.03,3437.74,3606.77,3624.78,3867.96,4875.77
2,ALUAR,ARALUA010258,1547.04,1820.11,1911.12,1866.11,1835.93,1805.22,1657.42,1911.95,...,3095.40,3702.96,3344.23,3256.09,3037.93,2793.77,3782.29,4021.72,3812.33,4041.31
3,BANCO BBVA ARGENTINA,ARP125991090,418.23,396.54,467.49,418.37,350.66,352.52,401.75,404.48,...,993.48,952.87,805.06,775.54,607.67,452.86,840.33,834.27,938.53,1039.75
4,TERNIUM ARGENTINA SOCIEDAD ANONIMA,ARSIDE010029,178.40,177.53,184.55,184.16,171.05,112.65,120.55,124.93,...,587.94,663.26,588.27,590.42,490.39,476.31,568.86,630.98,527.52,552.86
5,STRABAG SE,AT000000STR1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,430.27,430.83,479.46,477.35,465.06,461.50,396.65,452.87,481.49,526.88


In [48]:
# Compute simple returns and handle delisting
returns = prices[["NAME", "ISIN"]].copy()
returns[price_cols] = np.nan

for idx in prices.index:
    p = prices.loc[idx, price_cols]

    # simple returns, no forward-filling of prices
    r = p.pct_change(fill_method=None)

    # find the last valid price position
    valid_positions = p.notna()
    if valid_positions.any():
        last_valid_pos = valid_positions.values.nonzero()[0][-1]
        after_last = p.iloc[last_valid_pos + 1:]
        # if everything after last valid price is NaN -> delisted
        if len(after_last) > 0 and after_last.isna().all():
            r.iloc[last_valid_pos + 1] = -1.0
            # rest stay NaN

    returns.loc[idx, price_cols] = r.values

returns.to_excel("../data_clean/returns_monthly_clean.xlsx", index=False)
print("returns:", returns.shape)
returns.head()

/var/folders/rz/7_d0cf451mq8cgp_vjmwd0v40000gn/T/ipykernel_18467/359534553.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  returns[price_cols] = np.nan
/var/folders/rz/7_d0cf451mq8cgp_vjmwd0v40000gn/T/ipykernel_18467/359534553.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  returns[price_cols] = np.nan
/var/folders/rz/7_d0cf451mq8cgp_vjmwd0v40000gn/T/ipykernel_18467/359534553.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performa

returns: (2545, 316)


,NAME,ISIN,1999-12-31 00:00:00,2000-01-31 00:00:00,2000-02-29 00:00:00,2000-03-31 00:00:00,2000-04-28 00:00:00,2000-05-31 00:00:00,2000-06-30 00:00:00,2000-07-31 00:00:00,...,2025-04-30 00:00:00,2025-05-30 00:00:00,2025-06-30 00:00:00,2025-07-31 00:00:00,2025-08-29 00:00:00,2025-09-30 00:00:00,2025-10-31 00:00:00,2025-11-28 00:00:00,2025-12-31 00:00:00,2026-01-30 00:00:00
1,SLB,AN8068571086,NaN,0.087968,0.213043,0.035756,0.000814,-0.039184,0.017198,-0.009209,...,-0.204544,-0.006017,0.031463,0.000000,0.089941,-0.059450,0.049169,0.004993,0.067088,0.260553
2,ALUAR,ARALUA010258,NaN,0.176511,0.050002,-0.023552,-0.016173,-0.016727,-0.081874,0.153570,...,-0.320818,0.196278,-0.096877,-0.026356,-0.067001,-0.080371,0.353830,0.063303,-0.052065,0.060063
3,BANCO BBVA ARGENTINA,ARP125991090,NaN,-0.051861,0.178923,-0.105072,-0.161842,0.005304,0.139652,0.006795,...,-0.071956,-0.040877,-0.155121,-0.036668,-0.216456,-0.254760,0.855607,-0.007211,0.124972,0.107850
4,TERNIUM ARGENTINA SOCIEDAD ANONIMA,ARSIDE010029,NaN,-0.004877,0.039543,-0.002113,-0.071188,-0.341421,0.070129,0.036333,...,-0.321595,0.128108,-0.113063,0.003655,-0.169422,-0.028712,0.194306,0.109201,-0.163967,0.048036
5,STRABAG SE,AT000000STR1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.230679,0.001302,0.112875,-0.004401,-0.025746,-0.007655,-0.140520,0.141737,0.063197,0.094270
